# Triplet extraction with LLMs

`narrativegraphs` ships with a component that takes natural language instructions for doing triplet extraction of the entities and relations that you are interested in. An `LlmTripletExtractor` is created with instructions and passed to the `NarrativeGraph` object on creation.

In [5]:
import re
from gutenbergpy.textget import get_text_by_id, strip_headers

# first, let us fetch something as documents
raw_text = get_text_by_id(1342)  # Pride & Prejudice by Jane Austen
clean_text = strip_headers(raw_text)
decoded_text = clean_text.decode("utf-8")
splitter = re.compile(r"\n{4}(?:Chapter|CHAPTER) [IVX]+\.")
chapters = splitter.split(decoded_text)[1:]
ids, paragraphs = zip(*[(f"{i}-{j}", p) for i, c in enumerate(chapters)
                       for j, p in enumerate(c.split("\n\n"))][:30])

In [9]:
from narrativegraphs import NarrativeGraph
from narrativegraphs.nlp.triplets import LlmTripletExtractor
from narrativegraphs.nlp.common.llm import OpenAiCompatibleClient

extractor = LlmTripletExtractor(
    """The text is a paragraph from Pride & Prejudice by Jane Austen.

    Extract only character names as subjects and objects. Predicates must be verbs or contain verbs.""",
    llm=OpenAiCompatibleClient(model="qwen/qwen3.5-9b", base_url="http://127.0.0.1:1234/v1")  # Local LM Studio
)
model = NarrativeGraph(
    triplet_extractor=extractor,
)
model.fit(paragraphs, doc_ids=ids)

INFO:narrativegraphs.pipeline:Adding 30 documents to database
INFO:narrativegraphs.pipeline:Extracting triplets


Extracting triplets:   0%|          | 0/30 [00:00<?, ?it/s]

INFO:narrativegraphs.pipeline:Resolving entities and predicates
INFO:narrativegraphs.pipeline:Mapping triplets and tuplets
INFO:narrativegraphs.pipeline:Calculating stats


In [11]:
model.serve_visualizer()

INFO:     Started server process [47167]
INFO:     Waiting for application startup.
INFO:root:Database engine provided to state before startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [47167]
INFO:root:Server stopped
